# MECHA in metafspm, 9: the solutes' options

`MechaSolutes` moves a solute (sucrose, with starch in the cells) through the section's liquid network: diffusion
in the walls, the plasmodesmata and across the membranes, advection with the water (× (1 − σ) across a membrane),
reactions, sources and sinks (`doc/equations.rst`, Solutes). Its options, from MECHA's `reactions` and `develop`
branches:

| option | parameter |
|---|---|
| the network the solute moves in: everywhere, the apoplast only, the symplast only | `solute_pathway` 1, 2, 3 |
| the discretisation: Scharfetter–Gummel (exact for advection–diffusion along an edge) or upwind | `solute_scheme` 1, 2 |
| the time step: implicit Euler, Crank–Nicolson, explicit | `solute_theta` 1, 0.5, 0 |
| what is solved: one time step, the steady state coupled with the water, the steady state on given flows | `solute_solve` 1, 2, 3 |
| the concentrations held: the needle's sinks (phloem loading, xylem) at `c_sink`, and any node given by the experiment | `solute_sinks`, `fixed_sucrose` |
| storage, loss, liquid boundary | tutorial 6 |

Compared with the legacies in `test_standard_solutes.py`, `test_solutes.py`, `test_coupled_steady.py`.
**Run time:** about a minute.

> **Developer tutorial 9** of ten:
> | | notebook | what it covers |
> |---|---|---|
> | 0 | `00_overview.ipynb` | the package's modules, the original MECHA and its reactions branch, how the port is validated, the docs |
> | 1 | `01_granap_mecha.ipynb` | GRANAP's anatomies and MECHA's hydraulics coupled in a plant model; MECHA's inputs by role; scenarios; the barriers |
> | 2 | `02_soil_gradient.ipynb` | the plant problem with a soil drier on one side of a root section; a root partly out of contact, by position or by cells |
> | 3 | `03_needle_solutes.ipynb` | a pine needle of three sections, its transpiration, MechaSolutes over a day |
> | 4 | `04_full_plant.ipynb` | a whole seedling in its soil, run to the fixed point |
> | 5 | `05_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |
> | 6 | `06_solute_storage_and_decay.ipynb` | the solutes' storage fractions, their first-order loss and the liquid boundary |
> | 7 | `07_section_options.ipynb` | the section's hydraulic options: the barrier codes 0-11, φ-thickenings, aquaporins, plasmodesmata rules, walls, xylem pieces, tracheids, measured axial conductances |
> | 8 | `08_standard_problem.ipynb` | the standard problem's boundaries (soil sides, xylem and phloem pressures or flows, the elongation zone), the osmotic scenarios, the flows by layer |
> | 9 | `09_solute_options.ipynb` | the solutes' pathways, schemes and time steps, held concentrations and steady states, the coupled water and sugar |
>
> New to MECHA? Start with the user tutorials (`../user/01_first_run.ipynb`): they solve a root section and explain
> its results without the framework's details.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openalea.metafspm.coupling.composite_wrapper import CompositeModel
from openalea.metafspm.coupling.translator import Translator
from openalea.metafspm.scene.scene import Scene

from openalea.granap.metafspm.architectures import RootAxis
from openalea.granap.metafspm.component import GranapAnatomy, junction_rules
from openalea.mecha.component import MechaHydraulics
from openalea.mecha.solutes import MechaSolutes


class RootSugars(CompositeModel):                      # tutorial 6's plant model
    initiators = (RootAxis, GranapAnatomy)
    from_scale, nodes, wiring = "SubOrgan", "Compartment", junction_rules

    def __init__(self, data_structure, time_step):
        self.anatomy = GranapAnatomy(data_structure=data_structure, time_step=time_step)
        self.hydraulics = MechaHydraulics(data_structure=data_structure, time_step=time_step)
        self.solutes = MechaSolutes(data_structure=data_structure, time_step=time_step)
        self.declare_data_and_couple_components(translator_path=Translator(),
                                                components=(self.anatomy, self.hydraulics, self.solutes))

    def run(self):
        self.anatomy()
        self.hydraulics()
        self.solutes()


def root(time_step=3600., **parameters):
    """A scene of one root section (barrier 1), the solutes on its standard problem's flows (solute_flows 2: the
    soil at 0 hPa, the xylem at −5000 hPa), with *parameters*; returns (scene, DataStructure, MechaSolutes)."""
    scenario = dict(solute_flows=2., **parameters)
    scene = Scene(pd.DataFrame([dict(plant="root", model=RootSugars, x=0., y=0., z=0., rotation=0.,
                                     scenario={"parameters": scenario})]), time_step=time_step)
    population = scene.populations[0]
    return scene, population.data_structure, population.instance.solutes


def distance_to_centre(ds):
    x, y = ds.get("section_x"), ds.get("section_y")
    cells = ds.get("kind") == 0
    return np.hypot(x - x[cells].mean(), y - y[cells].mean()) * 1e3                     # mm -> µm

## 1. The pathways

A pulse of 50 µmol cm⁻³ in the cortex cells and in the walls of the section's left half; one hour. In the whole
network (1), the solute spreads through walls and cells; in the apoplast only (2), only the walls' pulse moves,
along the walls; in the symplast only (3), only the cells', through the plasmodesmata:

In [2]:
PULSE = 50e-6
results = {}
for pathway, name in ((1., "whole network"), (2., "apoplast only"), (3., "symplast only")):
    scene, ds, solutes = root(solute_pathway=pathway)
    kind, cgroup = ds.get("kind"), np.asarray(ds.get("cgroup"))
    x = ds.get("section_x")
    start = np.where((kind == 0) & (cgroup == 4), PULSE, 0.)
    start = np.where((kind > 0) & (x < x[kind == 0].mean()), PULSE, start)
    ds.set("sucrose", start)
    scene.run()
    moved = ds.get("sucrose") - start
    results[name] = pd.Series({"walls' change (µmol cm-3, mean |Δc|)": abs(moved[kind > 0]).mean() * 1e6,
                               "cells' change (µmol cm-3, mean |Δc|)": abs(moved[kind == 0]).mean() * 1e6})
pd.DataFrame(results).T

,"walls' change (µmol cm-3, mean |Δc|)","cells' change (µmol cm-3, mean |Δc|)"
whole network,23.506256,15.653895
apoplast only,26.799087,0.000000
symplast only,0.000000,11.669502


Each pathway alone isolates a transport route: the apoplast's diffusion and advection along the walls, or the
symplast's through the plasmodesmata. The membranes join the two in the whole network.

## 2. Schemes and time steps

With strong flows (high Péclet numbers: advection ≫ diffusion along an edge), the upwind scheme adds numerical
diffusion and Crank–Nicolson can oscillate; Scharfetter–Gummel with implicit Euler (the defaults) stays exact and
stable. A cortex pulse after one hour, with the diffusivities divided by 1000 (advection dominates):

In [3]:
slow = dict(d_apoplast=1e-4, d_plasmodesmata=5e-4, d_membrane=1e-9)
profiles = {}
for (scheme, theta), name in (((1., 1.), "Scharfetter–Gummel, implicit"), ((2., 1.), "upwind, implicit"),
                              ((1., 0.5), "Scharfetter–Gummel, Crank–Nicolson")):
    scene, ds, solutes = root(solute_scheme=scheme, solute_theta=theta, **slow)
    kind, cgroup = ds.get("kind"), np.asarray(ds.get("cgroup"))
    ds.set("sucrose", np.where((kind == 0) & (cgroup == 4), PULSE, 0.))
    scene.run()
    profiles[name] = pd.Series(ds.get("sucrose") * 1e6)
table = pd.DataFrame(profiles)
print(f"most negative concentration (µmol cm-3): {table.min().round(6).to_dict()}")
distance = distance_to_centre(ds)
for name in table:
    order = np.argsort(distance)
    plt.plot(distance[order], table[name].to_numpy()[order], ".", ms=2, label=name)
plt.xlabel("distance to the centre (µm)"); plt.ylabel("sucrose (µmol cm⁻³)"); plt.legend(markerscale=6)
plt.show()

most negative concentration (µmol cm-3): {'Scharfetter–Gummel, implicit': 0.0, 'upwind, implicit': 0.0, 'Scharfetter–Gummel, Crank–Nicolson': 0.0}


/tmp/ipykernel_1733854/2392258077.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The schemes differ where the flow is fast (the walls near the xylem); a negative concentration would show
Crank–Nicolson's oscillations at larger steps. Keep the defaults unless reproducing a legacy run.

## 3. Held concentrations and steady states

An experiment often holds a solute at a boundary: a tracer in the soil solution around the epidermis, a hormone
produced in some cells. `fixed_sucrose` (a node's input, NaN: free) holds those nodes at their value; `solute_sinks`
0 removes the needle's default sinks. `solute_solve` 3 solves the steady state on the water's flows directly
(`develop`'s tutorial 04). With a first-order loss `k_decay` (consumption, degradation), the solute penetrates over a
length that shrinks as the loss grows (√(D/k), the reaction–diffusion penetration depth):

In [4]:
profiles = {}
for k_decay in (0., 5., 48., 500.):
    scene, ds, solutes = root(solute_solve=3., solute_sinks=0., k_decay=k_decay)
    epidermis = (ds.get("kind") == 0) & (np.asarray(ds.get("cgroup")) == 2)
    ds.set("fixed_sucrose", np.where(epidermis, 1.0, np.nan))        # the epidermis held at 1 (a relative unit)
    scene.run()
    profiles[k_decay] = ds.get("sucrose").copy()
distance = distance_to_centre(ds)
cells = ds.get("kind") == 0
for k_decay, c in profiles.items():
    order = np.argsort(distance[cells])
    plt.plot(distance[cells][order], c[cells][order], ".", ms=3, label=f"k_decay {k_decay:g} d⁻¹")
plt.xlabel("distance to the centre (µm)"); plt.ylabel("steady concentration (held: 1)"); plt.legend()
plt.show()

/tmp/ipykernel_1733854/3663582533.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Without loss, the steady state is set by the water: the soil's clean water enters with the flow and the xylem's
leaves with the solute (the liquid boundary, tutorial 6), so the inner tissues stay below the epidermis' value. A
loss shapes a gradient from the held epidermis inward: at 48 d⁻¹ (MECHA's hormone default) little reaches the
stele.

## 4. The water and the sugar coupled

The sucrose's osmotic potential (−R T c, van 't Hoff) pulls water into the cells, and the water carries the
sucrose: `solute_solve` 2 solves both at once, at steady state (one Newton system per section; the reactions
branch's `coupled_water_solute_solve` by Picard). The membranes need a reflection coefficient (σ, the "Osmotic"
role: MechaOsmoticPresets, or here set directly, 0.7); sources in the cortex cells, on the standard problem's
water:

In [5]:
coupled = {}
for solve, name in ((3., "the water's flows given"), (2., "coupled")):
    scene, ds, solutes = root(solute_solve=solve)
    cortex = (ds.get("kind") == 0) & (np.asarray(ds.get("cgroup")) == 4)
    ds.set("sucrose_source", np.where(cortex, 1e-11, 0.))
    ds.set("reflection_coefficient", np.where(ds.get("network_path") == 0, 0.7, 0.))
    scene.run()
    psi = ds.get("coupled_psi") if solve == 2. else ds.get("standard_psi")
    coupled[name] = pd.Series({"cortex sucrose (µmol cm-3)": ds.get("sucrose")[cortex].mean() * 1e6,
                               "cortex water potential (hPa)": psi[cortex].mean()})
pd.DataFrame(coupled).T

,cortex sucrose (µmol cm-3),cortex water potential (hPa)
the water's flows given,34.376422,-442.945999
coupled,34.374573,153.262275


Coupled, the cortex's sugar lowers its cells' osmotic potential (−R T c: about −850 hPa at 34 µmol cm⁻³): water
flows in until their pressure potential (`coupled_psi`) has risen by about 600 hPa, turgor. The sucrose itself
barely changes: here the feedback acts on the water, not on the sugar's distribution (the water drawn in is small
next to the flow to the xylem).

## Try it yourself

1. Hold a hormone in two stele cells only (`fixed_sucrose` on them): where does it go at barrier 1 and at barrier 3?
2. At which `k_decay` does half the epidermis' value reach the endodermis?
3. Raise the sources tenfold: how far do the coupled and uncoupled water potentials part?